# 02 · Generate — two state backbones + multi-state MPNN shared-sequence pool

**Standard slot:** *design campaign.* **For Project 22 this means:** generate the **two state
backbones** (RFdiffusion ×2), then run **multi-state ProteinMPNN** (residue identities *tied* across
both backbones) to search for sequences compatible with **both** states, predict both states, and
assemble `results/multistate_designs.csv` (D2).

**Diversity before filtering** — generate a real pool here; you filter in notebook 03. The mock
backend runs anywhere; switch to the real backends on Colab/A100.

## Setup paths

In [ ]:
import sys, os
# Make the project's scripts/ and the cohort's shared/ importable.
# Adjust these if your Colab working directory differs (see 00_setup §5 for Drive mounting).
sys.path.insert(0, os.path.abspath("../scripts"))
sys.path.insert(0, os.path.abspath("../../../shared"))
os.makedirs("results", exist_ok=True)
print("paths ready; cwd =", os.getcwd())

## Version-verify the pinned upstreams (tools change)

The real backends live in fast-moving repos. Before a real run, confirm the pinned upstreams still
exist (HTTP `HEAD`). **Pin the exact commit/tag** in `scripts/multistate_tools.py` and `LOG.md` — the
status check below confirms reachability, *not* that your pinned commit is unchanged.

In [ ]:
import requests

# Pinned upstreams for the multi-state campaign (replace the comment with the COMMIT/TAG you pin):
UPSTREAMS = {
    "RFdiffusion": "https://github.com/RosettaCommons/RFdiffusion",   # pin a commit/tag in multistate_tools.py
    "ColabDesign": "https://github.com/sokrypton/ColabDesign",        # RFdiffusion + tied-MPNN driver; pin a commit
    "ProteinMPNN": "https://github.com/dauparas/ProteinMPNN",         # run in tied/multi-state mode; pin a commit
    "OpenMM":      "https://github.com/openmm/openmm",                # (extension) transition MD; pin a release
}
for name, url in UPSTREAMS.items():
    try:
        r = requests.head(url, allow_redirects=True, timeout=10)
        print(f"{name:12s} {r.status_code}  {url}")
    except Exception as e:  # noqa: BLE001
        print(f"{name:12s} UNREACHABLE  {url}  ({e})")
print("\n(Pin the exact commit/tag you use; log it. Reachability != version unchanged.)")

## 1 · Generate the two state backbones (RFdiffusion ×2)

State A and state B are two related-but-distinct backbones the **same** sequence must adopt. Two
common routes (see `MANUAL.md §2`): (a) generate A, then B as a conformational variant (partial
diffusion / hinge re-fold of A), or (b) build de novo backbones matching a known two-state template
pair (LOCKR latch/cage; an open/closed hinge). **This is two RFdiffusion runs — A100/HPC for the
campaign; a T4 runs only a tiny fallback.**

In [ ]:
import multistate_tools as ms

TOPOLOGY = "hinge"     # keep in sync with data/inputs/two_state_def.txt
TRIGGER  = "pH"
LENGTH   = 100
SEED     = 0

# tool="mock" -> placeholders (no GPU); on Colab use tool="rfdiffusion" (two backbone runs).
tsd = ms.generate_two_states(topology=TOPOLOGY, trigger=TRIGGER, tool="mock",
                             length=LENGTH, seed=SEED, out_dir="results/two_state")
print("state A:", tsd.state_a.as_row())
print("state B:", tsd.state_b.as_row())
print("trigger:", tsd.trigger, "-", tsd.trigger_detail)

## 2 · Multi-state ProteinMPNN — one sequence for BOTH backbones

Tie residue identities across the two backbones and optimize one shared sequence under **both**
states. Record the **per-state MPNN score** for A and B *separately* — a good switch fits BOTH, not
just one. Sample a pool (`N_DESIGNS`) so there is diversity to filter later. A single sequence
satisfying two states well is **rare**: expect a low yield (see `MANUAL.md §1`).

In [ ]:
N_DESIGNS = 24     # campaign scale; on a T4 fallback keep this small

# tool="mock" -> deterministic synthetic pool; on Colab use tool="proteinmpnn" (tied mode).
shared = ms.multistate_mpnn(tsd.state_a, tsd.state_b, n=N_DESIGNS, tool="mock", seed=SEED)
print(len(shared), "shared sequences (multi-state MPNN)")
for s in shared[:3]:
    print(" ", s.design_id, "| A score", s.mpnn_score_a, "| B score", s.mpnn_score_b,
          "| len", len(s.sequence))
print("[mock = SYNTHETIC EXAMPLE_DATA — not real designs]")

## 3 · Predict BOTH states for every shared sequence → results CSV

For each shared sequence, predict it toward **state A** and toward **state B** (AF2 for trusted
picks, ESMFold for triage), compute the per-state scRMSD/pLDDT, and the energy gap. Assemble one row
per design into `results/multistate_designs.csv` — the D2 artifact that notebooks 03/04 consume.

In [ ]:
import pandas as pd

rows = []
for s in shared:
    if not s.sequence:
        continue
    pa = ms.af2_predict_state(s.sequence, tsd.state_a, tool="mock")   # -> esmfold/af2 on Colab
    pb = ms.af2_predict_state(s.sequence, tsd.state_b, tool="mock")
    eg = ms.energy_gap(pa, pb)
    rows.append(dict(
        design_id=s.design_id, sequence=s.sequence, topology=TOPOLOGY, trigger=TRIGGER, seed=SEED,
        mpnn_score_a=s.mpnn_score_a, mpnn_score_b=s.mpnn_score_b,
        scrmsd_a=pa.scrmsd_to_state, plddt_a=pa.plddt,
        scrmsd_b=pb.scrmsd_to_state, plddt_b=pb.plddt,
        energy_gap=eg.gap, favored_state=eg.favored_state, switchable=eg.switchable,
        synthetic=bool(s.synthetic), tool="mock"))

df = pd.DataFrame(rows)
df.to_csv("results/multistate_designs.csv", index=False)
print("wrote results/multistate_designs.csv", df.shape)
df.head()

### A first look at the A-vs-B trade-off
Even on synthetic data you should see the prior the real problem has: sequences that fit A well tend
to fit B worse, only a few balance both, and the gap is small/noisy. Quantify how often *anything*
satisfies both states — that fraction is the story of the interim report.

In [ ]:
import numpy as np
pass_a = (df["scrmsd_a"] < ms.SELF_CONSISTENT_SCRMSD) & (df["plddt_a"] >= ms.SWITCH_PLDDT)
pass_b = (df["scrmsd_b"] < ms.SELF_CONSISTENT_SCRMSD) & (df["plddt_b"] >= ms.SWITCH_PLDDT)
print(f"N total              : {len(df)}")
print(f"pass state A (mock)  : {int(pass_a.sum())}")
print(f"pass state B (mock)  : {int(pass_b.sum())}")
print(f"pass BOTH (mock)     : {int((pass_a & pass_b).sum())}")
print(f"switchable gap (mock): {int(df['switchable'].fillna(False).sum())}")
print("\n[SYNTHETIC EXAMPLE_DATA — illustrates the trade-off, not a real hit rate]")

## D2 checklist
- [ ] Two state backbones generated (RFdiffusion ×2) — note the GPU (A100/HPC for the real run).
- [ ] `results/multistate_designs.csv`: one row per design with per-state scrmsd/pLDDT, per-state MPNN scores, energy gap, seed.
- [ ] Per-state MPNN scores recorded for **A and B separately** (the trade-off).
- [ ] Version-verify cell run; pinned commits/tags + seeds in `LOG.md`.
- [ ] 3–4 page interim report on the early A-vs-B trade-off and how often anything satisfies both states.

**Next:** `03_filter_and_rank.ipynb` — run the shared filter on **both** states.